## Analyze A/B Test Results



## Table of Contents

<a id='intro'></a>
### Introduction

A/B tests are very commonly performed by data analysts and data scientists.  It is important that you get some practice working with the difficulties of these test results.

For this project, you will be working to understand the results of an A/B test run by an e-commerce website.  Your goal is to work through this notebook to help the company understand if they should implement the new page, keep the old page, or perhaps run the experiment longer to make their decision. For a user to "convert" means that they have decided to purchase the company's product.

<a id='probability'></a>
### Part I - Probability

To get started, let's import our libraries.

In [ ]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
%matplotlib inline
#We are setting the seed to assure you get the same answers on quizzes as we set up
random.seed(42)

In [ ]:
df = pd.read_csv('ab_data.csv')
df.head(20)

In [ ]:
df_rows = df.shape[0]
df_rows

In [ ]:
df_nunique_user = df.user_id.nunique()
df_nunique_user

In [ ]:
df_nunique_user * 100 / df_rows

In [ ]:
df.head()

In [ ]:
df_prop_converted = df.converted.mean()
df_prop_converted

Identifying errors in webpage

In [ ]:
#count instances comparing landing_page to group
print(df.groupby(['landing_page', 'group']).count())

#new_page and control OR old_page and treatment
df_num_no_match = 1928 + 1965
df_num_no_match

In [ ]:
df.groupby(['landing_page', 'group']).count()

Check Missing values

In [ ]:
df.isna().sum().max()

Drop errors in webpage

In [ ]:
#drop rows where (group == treatment & landing_page == old_age) OR
#drop rows where (group == control & landing_page == new_page)
df_rows_drop = df.query('((group == "treatment" and landing_page == "old_page")) or ((group == "control" and landing_page == "new_page"))')

#store in new DataFrame
df2 = df.drop(df_rows_drop.index)
df2.shape[0]

In [ ]:
df2.groupby(['landing_page', 'group']).count()

In [ ]:
# Double Check all of the correct rows were removed - this should be 0
df2[((df2['group'] == 'treatment') == (df2['landing_page'] == 'new_page')) == False].shape[0]

Check Number of unique users

In [ ]:
df2_nunique_user = df2.user_id.nunique()
df2_nunique_user

Check replicated users

In [ ]:
repeated = df2.groupby( 'user_id' ).size().reset_index( name = 'count' )
repeated[ repeated[ 'count' ] > 1 ]

# Drop duplications

df2.drop_duplicates( subset= 'user_id', inplace = True )

In [ ]:
# Probability of converting regardless the page they recieve
df2.converted.mean()

Probability of converting in the `control` group

In [ ]:
df2_control = df2[ df2[ 'group' ] == 'control' ]
df2_control.converted.mean()

Probability of converting in the `treatment` group

In [ ]:
df2_treatment = df2[df2['group'] == 'treatment']
df2_treatment.converted.mean()

Probability that an individual received the new page

In [ ]:
df2.groupby( 'landing_page' ).size() / df2.shape[0]

<a id='ab_test'></a>
### Part II - A/B Test


|**$p_{old}$** | **$p_{new}$** |
|--|--|
|the converted rates for the old  pages. | the converted rates for the new  pages.

***Words***

Null hypothesis: Individuals with the new page have an **equal or worse** conversion rate as individuals with the old page.

Alternative hypothesis: Individuals with the new page have a **higher** conversion rate than individuals with the old page.

***Notation***

$H_0: p_{new} - p_{old} \leq 0$

$H_1: p_{new} - p_{old} > 0$

`2.` Assume under the null hypothesis, $p_{new}$ and $p_{old}$ both have "true" success rates equal to the **converted** success rate regardless of page - that is $p_{new}$ and $p_{old}$ are equal. Furthermore, assume they are equal to the **converted** rate in **ab_data.csv** regardless of the page. <br><br>

Use a sample size for each page equal to the ones in **ab_data.csv**.  <br><br>

Perform the sampling distribution for the difference in **converted** between the two pages over 10,000 iterations of calculating an estimate from the null.  <br><br>

Use the cells below to provide the necessary parts of this simulation.  If this doesn't make complete sense right now, don't worry - you are going to work through the problems below to complete this problem.  You can use **Quiz 5** in the classroom to make sure you are on the right track.<br><br>

In [ ]:
pnew = df2.converted.mean()
pnew

In [ ]:
#p_{old} = p_{new} under the null
pold = df2.converted.mean()
pold

In [ ]:
nnew = df2[df2['group'] == 'treatment'].shape[0]
nnew

In [ ]:
nold = df2[df2['group'] == 'control'].shape[0]
nold

In [ ]:
# Simulate $n_{new}$ transactions with a convert rate of $p_{new}$ under the null. 
# Store these $n_{new}$ 1's and 0's in **new_page_converted**.
new_page_converted = np.random.normal(0, pnew, nnew)
new_page_converted

In [ ]:
old_page_converted = np.random.normal(0, pold, nold)

In [ ]:
obs_diff = df2_treatment.converted.mean() - df2_control.converted.mean()
obs_diff

In [ ]:
nnew

In [ ]:
nold

In [ ]:
p_diffs = []

# Numbre of trials, number of exists and number of flips
new_page = np.random.binomial( nnew, pnew, 10000 )/nnew
old_page = np.random.binomial( nold, pold, 10000 )/nold

p_diffs = new_page - old_page

In [ ]:
plt.hist(p_diffs);
plt.axvline( x = obs_diff, color='red');

In [ ]:
(p_diffs > obs_diff).mean()


Since p = 0.9017 is significantly greater than the threshold for Type I error rates at 0.05, we fail to reject the null hypothesis that: $$p_{new} \leq p_{old}$$

$$or$$

$$p_{new} - p_{old} \leq 0$$

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.proportion import proportions_ztest as ztest

#number of conversions for each page
convert_old = df2[df2['landing_page'] == 'old_page'].converted.sum()
convert_new = df2[df2['landing_page'] == 'new_page'].converted.sum()

#number of individuals who received each page
n_old = df2.query('landing_page == "old_page"').shape[0]
n_new = df2.query('landing_page == "new_page"').shape[0]

In [ ]:
zstat, pval = ztest([convert_new, convert_old],
                 [n_new, n_old],
                 alternative='larger')

print(zstat, pval)

The z-score calculated here is **-1.3109**. This means that the observed difference in conversion rates (-0.001578) between $p_{new}$ and $p_{old}$ is 1.3109 standard deviations away from the mean, which is 0. Let us examine the plot again.

In [ ]:
plt.hist(p_diffs);
plt.axvline(x=obs_diff, color='red');

<a id='regression'></a>
### Part III - A regression approach

Logistic regression.

In [ ]:
#create intercept, map to new df
df2a = df2.copy()
df2a['intercept'] = 1

#create dummy variables for landing_page, map to new df
ab_dummies = pd.get_dummies(df2a['landing_page'])
df2b = df2a.join(ab_dummies)
df2b.rename(columns = {'new_page' : 'ab_page'}, inplace=True)
df2b.head()

In [ ]:
#drop baseline variable, 'old_page'
df2b.drop('old_page', axis = 1, inplace = True)
df2b.head()

In [ ]:
import statsmodels.api as sm

rm = sm.Logit(df2b['converted'], df2b[['intercept', 'ab_page']])

In [ ]:
rm.fit().summary()

In [ ]:
sm.OLS( df2b['converted'], df2b[['intercept', 'ab_page']] ).fit().summary()



Does it appear that country had an impact on conversion?  Don't forget to create dummy variables for these country columns - **Hint: You will need two columns for the three dummy variables.** Provide the statistical output as well as a written response to answer this question.

In [ ]:
countries_df = pd.read_csv('./countries.csv')
df_new = countries_df.set_index('user_id').join(df2.set_index('user_id'), how='inner')

df_new.head()

In [ ]:
### Create the necessary dummy variables
country_dummies = pd.get_dummies(df_new['country'])

df_newa = df_new.join(country_dummies)
df_newa.head()

In [ ]:
#'US' as baseline country
df_newa['intercept'] = 1
rm_a = sm.OLS(df_newa['converted'], df_newa[['intercept', 'CA', 'UK']])
rm_a.fit().summary()

In [ ]:
#exponentiate coefficients
CA = np.exp(-0.0408)
UK = 1/np.exp(0.0099)

UK, CA

In [ ]:
#get page dummies
page_dummies = pd.get_dummies(df_new['landing_page'])
page_dummies.head()

In [ ]:
#form new df with all necessary columns
df_newb = df_newa.join(page_dummies)
df_newb.head()

In [ ]:
df_newb = df_newb.rename(columns = {'new_page' : 'ab_page'} )
df_newb.drop('old_page', axis = 1, inplace = True)
df_newb.head()

In [ ]:
#create interaction column
df_newc = df_newb.copy()
df_newc['intercept'] = 1
df_newc['UK_ab_page'] = df_newb['UK'] * df_newb['ab_page']
df_newc['CA_ab_page'] = df_newb['CA'] * df_newb['ab_page']
df_newc.head()

In [ ]:
#logistic regression model with US and old_page as baseline
rm_b = sm.Logit(df_newc['converted'], df_newc[['intercept', 'UK', 'CA', 'ab_page', 'UK_ab_page', 'CA_ab_page']])
rm_b.fit().summary()

In [ ]:
#exponentiate coefficients

UK2 = np.exp(-.0046)
CA2 = np.exp(-0.0175
            )
ab_page = np.exp(-0.0236)
UK_ab_page = np.exp(-0.0236)
CA_ab_page = np.exp(-0.0469)

UK2, CA2, ab_page, UK_ab_page, CA_ab_page

In [ ]:
from statsmodels.stats.power import TTestPower

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.power import TTestIndPower
  

# power analysis varying parameters
effect_sizes = np.array([0.2, 0.5, 0.8,1.3])
sample_sizes = np.array(range(5, 100))
  
# plot power curves
obj = TTestIndPower()
obj.plot_power(dep_var = 'nobs', 
               nobs = sample_sizes,
               effect_size = effect_sizes)
  
plt.show()